# 01 — Data Inspection

## Purpose



This notebook performs the initial inspection, cleaning, and validation of the
World Bank Enterprise Survey (WBES) datasets used in the thesis.

The analysis uses:

- Kenya — WBES 2025
- Tanzania — WBES 2023
- Uganda — WBES 2025

The workflow in this notebook is:

Raw WBES data
→ variable inspection
→ coding inspection
→ missing-value identification
→ data cleaning
→ target construction
→ predictor validation
→ final quality checks
→ cleaned datasets

The cleaned datasets produced by this notebook will be used as the input for
`02_descriptive_analysis.ipynb`.

No train/test split, SMOTE, scaling, model fitting, or hyperparameter tuning
is performed in this notebook.


In [31]:

# IMPORT LIBRARIES


from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyreadstat
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 1. Project Paths

The raw WBES datasets are stored separately from the cleaned datasets.

The raw files are never overwritten.

Cleaned versions will be saved in:

`../Data/clean/`

This allows the original WBES files to remain unchanged and provides a
reproducible cleaning pipeline.

In [17]:

# PROJECT PATHS


PROJECT_DIR = Path("..")

DATA_DIR = PROJECT_DIR / "Data"

RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"

RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"

CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:")
print(PROJECT_DIR.resolve())

print("\nRaw data directory:")
print(RAW_DIR.resolve())

print("\nClean data directory:")
print(CLEAN_DIR.resolve())

print("\nResults directory:")
print(RESULTS_DIR.resolve())

Project directory:
/Users/eddiebest/Documents/SME_Export_Thesis

Raw data directory:
/Users/eddiebest/Documents/SME_Export_Thesis/Data/raw

Clean data directory:
/Users/eddiebest/Documents/SME_Export_Thesis/Data/clean

Results directory:
/Users/eddiebest/Documents/SME_Export_Thesis/results


## 2. Define the Raw WBES Datasets

The analysis uses three country-level WBES datasets:

- Kenya
- Tanzania
- Uganda

The raw files are used as the starting point for the cleaning process.

In [26]:

# RAW WBES FILES


countries = [
    "Kenya",
    "Tanzania",
    "Uganda"
]

raw_paths = {
    "Kenya": RAW_DIR / "Kenya-2025-full-data.dta",
    "Tanzania": RAW_DIR / "Tanzania-2023-full-data.dta",
    "Uganda": RAW_DIR / "Uganda-2025-full-data.dta"
}

for country in countries:

    print(
        country,
        " ",
        raw_paths[country]
    )

Kenya   ../Data/raw/Kenya-2025-full-data.dta
Tanzania   ../Data/raw/Tanzania-2023-full-data.dta
Uganda   ../Data/raw/Uganda-2025-full-data.dta


## 3. Verify the Raw Files

Before loading the datasets, verify that all three expected WBES files exist.

If a file is missing, the notebook stops rather than silently using a different
dataset.

In [20]:

# RAW FILE EXISTENCE CHECK


for country in countries:

    path = raw_paths[country]

    print(
        f"{country}:",
        path.exists()
    )

    assert path.exists(), (
        f"Raw WBES file not found: {path.resolve()}"
    )

print("\nAll required raw WBES files were found.")

Kenya: True
Tanzania: True
Uganda: True

All required raw WBES files were found.


## 4. Load the Raw WBES Datasets

The raw `.dta` files are loaded without modifying their values.

At this stage the datasets are stored exactly as supplied by the WBES files.

In [21]:

# LOAD RAW DATASETS


raw_data = {}

for country in countries:

    raw_data[country] = pd.read_stata(
        raw_paths[country],
        convert_categoricals=True
    )

    print(
        f"{country}:",
        raw_data[country].shape
    )

Kenya: (1024, 336)
Tanzania: (600, 386)
Uganda: (605, 336)


## 5. Initial Dataset Dimensions

Inspect the number of observations and variables in each raw dataset.

This provides the baseline against which the cleaned datasets will later be
compared.

In [22]:

# INITIAL RAW DATASET DIMENSIONS


raw_dimensions = []

for country in countries:

    df = raw_data[country]

    raw_dimensions.append({
        "Country": country,
        "Rows": len(df),
        "Columns": df.shape[1]
    })

raw_dimensions = pd.DataFrame(
    raw_dimensions
)

display(raw_dimensions)

,Country,Rows,Columns
0,Kenya,1024,336
1,Tanzania,600,386
2,Uganda,605,336


## 6. Inspect Raw Variable Names

Display all variable names in each raw WBES dataset.

This is important because the same conceptual variable must be correctly
identified before any cleaning or harmonisation is performed.

In [25]:

# RAW VARIABLE NAMES


for country in countries:

    print("\n" + "  " * 70)
    print(country)
   

    print(
        raw_data[country].columns.tolist()
    )


                                                                                                                                            
Kenya
['idstd', 'id', 'a4a', 'a6a', 'a2', 'a1c', 'a0', 'a3a', 'a6c', 'a7', 'panel', 'a1a', 'a14d', 'a14m', 'a14y', 'a14h', 'a14min', 'a20y', 'a20m', 'a20d', 'a1', 'a12', 'a7a', 'a7b', 'a11', 'a7c', 'a9', 'b1', 'b1x', 'b3a', 'b2a', 'b2b', 'b2c', 'b2d', 'b4', 'b4a', 'b5', 'b6', 'b6b', 'b7', 'b7a', 'b8', 'b8x', 'c3', 'c4', 'c5', 'c31', 'c6', 'c7', 'c8a', 'c8b', 'c9a', 'c9b', 'c10', 'c11', 'c43', 'c44', 'c12', 'c13', 'c14', 'c33', 'c45', 'c152', 'c162', 'c172', 'c22b', 'c36', 'c37', 'c39', 'c42', 'd1a1a', 'd1a3', 'd2', 'd2x', 'n3', 'n3x', 'f1', 'd3a', 'd3b', 'd3c', 'd31x', 'd42', 'd43a', 'd43b', 'd4a', 'd4b', 'd342', 'd352', 'd8', 'd12a', 'd12b', 'd13', 'd38x', 'd44', 'd45a', 'd45b', 'd14a', 'd14b', 'd412', 'o1', 'o2', 'o3a2', 'o3b2', 'j31', 'r1', 'r2', 'r3', 'r4', 'r5', 'r6', 'r7', 'r8', 'r9', 'r10', 'r11', 'e1', 'e2b', 'e312', 'e32', 'e33', 'e6', '

## 7. Identify the TVariables

The current  modelling specification uses:

### Dependent variable

`exporter`

The exporter outcome will be constructed from the WBES direct-export variable
after its raw coding has been inspected.

### Predictors

Numeric:

- `l1`
- `d2`
- `b7`
- `b2b`

Binary:

- `h1`
- `c22b`
- `c36`
- `e6`
- `b3a`
- `l10`
- `c39`

Ordinal:

- `k30`

These variables will not be cleaned blindly. Their raw WBES labels and response
codes will first be inspected.

In [27]:

# THESIS VARIABLE SPECIFICATION


target_source = "d3c"

numeric_variables = [
    "l1",
    "d2",
    "b7",
    "b2b"
]

binary_variables = [
    "h1",
    "c22b",
    "c36",
    "e6",
    "b3a",
    "l10",
    "c39"
]

ordinal_variables = [
    "k30"
]

predictors = (
    numeric_variables
    + binary_variables
    + ordinal_variables
)

required_variables = [
    target_source
] + predictors

print("Target source:")
print(target_source)

print("\nNumeric variables:")
print(numeric_variables)

print("\nBinary variables:")
print(binary_variables)

print("\nOrdinal variables:")
print(ordinal_variables)

print("\nTotal predictors:")
print(len(predictors))

print("\nRequired raw variables:")
print(required_variables)

Target source:
d3c

Numeric variables:
['l1', 'd2', 'b7', 'b2b']

Binary variables:
['h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39']

Ordinal variables:
['k30']

Total predictors:
12

Required raw variables:
['d3c', 'l1', 'd2', 'b7', 'b2b', 'h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39', 'k30']


## 8. Verify Required Variables

Check that all variables required by the thesis are available in every country
dataset.


In [29]:

# REQUIRED VARIABLE CHECK


for country in countries:

    df = raw_data[country]

    missing_variables = [
        variable
        for variable in required_variables
        if variable not in df.columns
    ]

    print("\n" + "  " * 70)
    print(country)
    

    print(
        "Missing required variables:",
        missing_variables
    )

    assert not missing_variables, (
        f"{country} is missing: {missing_variables}"
    )

print(
    "\nRequired-variable verification passed for all countries."
)


                                                                                                                                            
Kenya
Missing required variables: []

                                                                                                                                            
Tanzania
Missing required variables: []

                                                                                                                                            
Uganda
Missing required variables: []

Required-variable verification passed for all countries.


## 9. Inspect WBES Variable Labels

Before cleaning the data, inspect the WBES variable labels.

This prevents incorrect assumptions about what numeric codes represent.

The labels will be used to determine:

- valid responses;
- missing/special responses;
- binary coding;
- ordinal coding;
- the correct exporter threshold variable.

In [32]:

# EXTRACT WBES VARIABLE LABELS


label_tables = []

for country in countries:

    raw, meta = pyreadstat.read_dta(
        raw_paths[country],
        apply_value_formats=False
    )

    labels = pd.DataFrame({
        "variable": meta.column_names,
        "label": meta.column_labels
    })

    labels["country"] = country

    label_tables.append(labels)


variable_labels = pd.concat(
    label_tables,
    ignore_index=True
)

display(
    variable_labels[
        variable_labels["variable"].isin(
            required_variables
        )
    ]
)

,variable,label,country
29,b3a,Is the largest owner also the top manager,Kenya
31,b2b,"% Owned By Private Foreign Individuals, Compan...",Kenya
39,b7,How Many Years of Experience Working In This S...,Kenya
65,c22b,Establishment Has Its Own Website,Kenya
66,c36,Application To Obtain Fixed Broadband Internet...,Kenya
68,c39,Did You Experience Internet Disruptions In Las...,Kenya
72,d2,"In Last Fiscal Year, What Were This Establishm...",Kenya
79,d3c,% of Sales: Direct Exports,Kenya
120,e6,Do You Use Technology Licensed From A Foreign-...,Kenya
122,h1,New Products/Services Introduced Over Last 3 Yrs,Kenya


## 10. Inspect Raw Value 



For each thesis variable, inspect the actual raw WBES response categories.

This allows us to distinguish:

- valid observations;
- legitimate zero values;
- "don't know";
- "refused";
- "not applicable";
- other WBES special responses.



In [34]:

# RAW VALUE DISTRIBUTIONS


for country in countries:

    df = raw_data[country]

    
    print(country)
    

    for variable in required_variables:

        
        print(variable)

        display(
            df[variable]
            .value_counts(
                dropna=False
            )
            .head(30)
        )

Kenya
d3c


d3c
0      845
20      39
30      30
40      28
10      23
100      9
5        6
15       5
60       5
25       4
50       4
45       3
70       3
80       3
1        2
90       2
2        1
3        1
4        1
14       1
28       1
35       1
44       1
76       1
85       1
93       1
95       1
98       1
99       1
Name: count, dtype: int64

l1


l1
5      66
10     53
8      48
7      47
15     44
6      40
12     36
20     33
25     31
40     29
30     25
50     16
11     15
100    15
120    15
3      14
16     14
21     14
70     14
14     13
45     13
9      12
13     12
18     12
22     12
4      11
32     11
35     11
80     11
24     10
Name: count, dtype: int64

d2


...

l10
No     426
Yes    179
Name: count, dtype: int64

c39


c39
No                                                        289
The establishment does not have an internet connection    191
Yes                                                       124
Don't know (spontaneous)                                    1
Name: count, dtype: int64

k30


k30
No obstacle                 168
Minor obstacle              168
Major obstacle              118
Moderate obstacle            88
Very severe obstacle         60
Don't know (spontaneous)      2
Does not apply                1
Name: count, dtype: int64

## 11. Define WBES Special Responses

WBES datasets contain legitimate survey responses such as:

- "Don't know (spontaneous)"
- "Does not apply"

These responses do not represent valid measurements for the corresponding
variables and will therefore be converted to missing values during cleaning.

However, substantive responses such as:

- "No"
- "Yes"
- "The establishment does not have an internet connection"
- numeric percentages

must not be treated as missing.

The cleaning rules are therefore variable-specific.

In [35]:

# WBES SPECIAL RESPONSE LABELS


SPECIAL_RESPONSES = [
    "Don't know (spontaneous)",
    "Does not apply",
    "Don't know",
    "Refused"
]

print("Special responses to be treated as missing:")
for response in SPECIAL_RESPONSES:
    print("  ", response)

Special responses to be treated as missing:
   Don't know (spontaneous)
   Does not apply
   Don't know
   Refused


## 12. Create Working Copies

The raw datasets are preserved unchanged.

Separate working copies are created for cleaning.

This ensures that the original WBES data remain available for verification
and reproducibility.

In [36]:

# CREATE WORKING COPIES


working_data = {
    country: raw_data[country].copy()
    for country in countries
}

print("Working copies created for:")
print(", ".join(countries))

Working copies created for:
Kenya, Tanzania, Uganda


## 13. Clean the Exporter Source Variable

The exporter outcome is based on `d3c`, which records the percentage of sales
exported directly.

The binary outcome is defined as:

- `exporter = 1` if direct exports are greater than 0%;
- `exporter = 0` if direct exports equal 0%.

Non-substantive responses such as "Don't know (spontaneous)" are treated as
missing.

The original `d3c` variable is retained during cleaning for traceability.

In [37]:

# CLEAN EXPORTER SOURCE VARIABLE


for country in countries:

    df = working_data[country]

    # Preserve original d3c
    d3c_clean = df["d3c"].copy()

    # Convert special responses to missing
    d3c_clean = d3c_clean.replace(
        SPECIAL_RESPONSES,
        np.nan
    )

    # Convert numeric responses to numeric
    d3c_numeric = pd.to_numeric(
        d3c_clean,
        errors="coerce"
    )

    # Construct binary exporter outcome
    df["exporter"] = np.where(
        d3c_numeric.isna(),
        np.nan,
        np.where(
            d3c_numeric > 0,
            1,
            0
        )
    )

    # Use nullable integer representation
    df["exporter"] = (
        pd.Series(
            df["exporter"],
            index=df.index
        )
        .astype("Int64")
    )

    working_data[country] = df

## 14. Verify the Exporter Variable

The newly constructed `exporter` variable is checked for:

- non-exporters (`0`);
- exporters (`1`);
- missing observations.

This check is performed before any observations are removed.

In [38]:

# VERIFY EXPORTER


for country in countries:

    print("\n" + "  " * 70)
    print(country)
  

    print(
        working_data[country]["exporter"]
        .value_counts(dropna=False)
        .sort_index()
    )


                                                                                                                                            
Kenya
exporter
0    845
1    179
Name: count, dtype: Int64

                                                                                                                                            
Tanzania
exporter
0       504
1        86
<NA>     10
Name: count, dtype: Int64

                                                                                                                                            
Uganda
exporter
0    539
1     66
Name: count, dtype: Int64


## 15. Clean Binary Variables

The binary WBES variables are harmonised to:

- `1` = Yes
- `0` = No
- missing = non-substantive response

The cleaning is performed using the original labelled responses.

This avoids incorrectly interpreting special WBES responses as valid zeros.

In [ ]:

# CLEAN BINARY VARIABLES

for country in countries:

    df = working_data[country]

    for variable in binary_variables:

        # Preserve original series
        s = df[variable].copy()

        
        s = s.astype("object")

        
        # Standardise textual responses
        

        s = s.replace(
            SPECIAL_RESPONSES,
            np.nan
        )

        
        # Standardise text formatting
        

        s = s.apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

       
        # Explicit binary mapping
        

        s = s.replace({
            "Yes": 1,
            "No": 0,
            "yes": 1,
            "no": 0
        })

        
        # Convert numeric representations if present
       

        s = pd.to_numeric(
            s,
            errors="coerce"
        )

        
        # Keep only valid binary values
        

        s = s.where(
            s.isin([0, 1]),
            np.nan
        )

       
        # Store cleaned variable
        

        df[variable] = s

    working_data[country] = df

print("✓ Binary variables cleaned successfully.")

✓ Binary variables cleaned successfully.


In [43]:

# VERIFY BINARY VARIABLES


for country in countries:

    print("\n" + "  " * 70)
    print(country)
   

    for variable in binary_variables:

        values = set(
            working_data[country][variable]
            .dropna()
            .unique()
        )

        print(
            f"{variable}:",
            sorted(values)
        )

        assert values.issubset({0, 1}), (
            f"Unexpected values in {country} - {variable}: "
            f"{values}"
        )

print(
    "\nBinary-variable coding verification passed."
)


                                                                                                                                            
Kenya
h1: [np.float64(0.0), np.float64(1.0)]
c22b: [np.int64(0), np.int64(1)]
c36: [np.int64(0), np.int64(1)]
e6: [np.float64(0.0), np.float64(1.0)]
b3a: [np.int64(0), np.int64(1)]
l10: [np.int64(0), np.int64(1)]
c39: [np.float64(0.0), np.float64(1.0)]

                                                                                                                                            
Tanzania
h1: [np.float64(0.0), np.float64(1.0)]
c22b: [np.float64(0.0), np.float64(1.0)]
c36: [np.float64(0.0), np.float64(1.0)]
e6: [np.float64(0.0), np.float64(1.0)]
b3a: [np.float64(0.0), np.float64(1.0)]
l10: [np.float64(0.0), np.float64(1.0)]
c39: [np.float64(0.0), np.float64(1.0)]

                                                                                                                                            
Uganda
h1: [np.int64(0), np.int64

## 17. Special Treatment of Internet Access (`c39`)

The raw `c39` responses contain three substantive states:

- Yes;
- No;
- The establishment does not have an internet connection.

The last response is not a missing observation. It indicates that the
establishment has no internet connection.

For the modelling specification, `c39` will be represented as a binary
internet-access indicator:

- `1` = Yes, the establishment has an internet connection;
- `0` = No or no internet connection.

"Don't know (spontaneous)" is treated as missing.

This treatment preserves the substantive distinction between having and not
having internet access while producing a binary predictor suitable for the
modelling stage.

In [44]:

# CLEAN INTERNET ACCESS VARIABLE


for country in countries:

    df = working_data[country]

    s = df["c39"].copy()

    # Special non-substantive response
    s = s.replace(
        SPECIAL_RESPONSES,
        np.nan
    )

    # Binary internet-access coding
    s = s.replace({
        "Yes": 1,
        "No": 0,
        "The establishment does not have an internet connection": 0
    })

    s = pd.to_numeric(
        s,
        errors="coerce"
    )

    df["c39"] = s

    working_data[country] = df

## 18. Verify Internet Access Coding

The cleaned `c39` variable should contain:

- `1` = internet connection;
- `0` = no internet connection;
- missing = don't know or other non-substantive response.

In [45]:

# VERIFY c39


for country in countries:

    print("\n" + "  " * 70)
    print(country)
   

    print(
        working_data[country]["c39"]
        .value_counts(dropna=False)
        .sort_index()
    )

    values = set(
        working_data[country]["c39"]
        .dropna()
        .unique()
    )

    assert values.issubset({0, 1}), (
        f"Unexpected c39 values in {country}: {values}"
    )

print(
    "\nc39 verification passed."
)


                                                                                                                                            
Kenya
c39
0.0    418
1.0    542
NaN     64
Name: count, dtype: int64

                                                                                                                                            
Tanzania
c39
0.0    199
1.0    112
NaN    289
Name: count, dtype: int64

                                                                                                                                            
Uganda
c39
0.0    289
1.0    124
NaN    192
Name: count, dtype: int64

c39 verification passed.


## 19. Clean the Ordinal Variable (`k30`)

The raw `k30` responses represent the severity of obstacles faced by the
establishment.

The intended ordinal ordering is:

1. No obstacle
2. Minor obstacle
3. Moderate obstacle
4. Major obstacle
5. Very severe obstacle

The non-substantive responses:

- Don't know (spontaneous)
- Does not apply

are treated as missing.

The resulting variable is coded from 0 to 4 so that the ordering is preserved.

In [ ]:

# CLEAN ORDINAL VARIABLE: k30


k30_mapping = {
    "No obstacle": 0,
    "Minor obstacle": 1,
    "Moderate obstacle": 2,
    "Major obstacle": 3,
    "Very severe obstacle": 4
}

for country in countries:

    df = working_data[country]

    # Preserve original series
    s = df["k30"].copy()

   

    s = s.astype("object")

  

    s = s.replace(
        SPECIAL_RESPONSES,
        np.nan
    )

    

    s = s.replace(
        k30_mapping
    )

    
    s = pd.to_numeric(
        s,
        errors="coerce"
    )

   
    # Store cleaned variable
   
    df["k30"] = s

    working_data[country] = df

print("✓ k30 cleaned successfully.")

✓ k30 cleaned successfully.


## 20. Verify Ordinal Coding

The cleaned `k30` variable should contain only the ordered values:

- `0` = No obstacle
- `1` = Minor obstacle
- `2` = Moderate obstacle
- `3` = Major obstacle
- `4` = Very severe obstacle

Missing values are retained as missing.

In [50]:

# VERIFY k30


for country in countries:

    print("\n" + "  " * 70)
    print(country)
    

    print(
        working_data[country]["k30"]
        .value_counts(dropna=False)
        .sort_index()
    )

    values = set(
        working_data[country]["k30"]
        .dropna()
        .unique()
    )

    assert values.issubset({
        0, 1, 2, 3, 4
    }), (
        f"Unexpected k30 values in {country}: {values}"
    )

print(
    "\nk30 ordinal coding verification passed."
)


                                                                                                                                            
Kenya
k30
0    154
1    253
2    257
3    209
4    151
Name: count, dtype: int64

                                                                                                                                            
Tanzania
k30
0.0     94
1.0     96
2.0    182
3.0    109
4.0    118
NaN      1
Name: count, dtype: int64

                                                                                                                                            
Uganda
k30
0.0    168
1.0    168
2.0     88
3.0    118
4.0     60
NaN      3
Name: count, dtype: int64

k30 ordinal coding verification passed.


## 19. Clean the Ordinal Variable (`k30`)

The raw `k30` responses represent the severity of obstacles faced by the
establishment.

The intended ordinal ordering is:

1. No obstacle
2. Minor obstacle
3. Moderate obstacle
4. Major obstacle
5. Very severe obstacle

The non-substantive responses:

- Don't know (spontaneous)
- Does not apply

are treated as missing.

The resulting variable is coded from 0 to 4 so that the ordering is preserved.

In [51]:

# CLEAN ORDINAL VARIABLE k30


k30_mapping = {
    "No obstacle": 0,
    "Minor obstacle": 1,
    "Moderate obstacle": 2,
    "Major obstacle": 3,
    "Very severe obstacle": 4
}

for country in countries:

    df = working_data[country]

    s = df["k30"].copy()

    # Remove non-substantive responses
    s = s.replace(
        SPECIAL_RESPONSES,
        np.nan
    )

    # Apply ordinal mapping
    s = s.replace(
        k30_mapping
    )

    # Convert to numeric
    s = pd.to_numeric(
        s,
        errors="coerce"
    )

    df["k30"] = s

    working_data[country] = df

## 20. Verify Ordinal Coding

The cleaned `k30` variable should contain only the ordered values:

- `0` = No obstacle
- `1` = Minor obstacle
- `2` = Moderate obstacle
- `3` = Major obstacle
- `4` = Very severe obstacle

Missing values are retained as missing.

In [52]:

# VERIFY k30


for country in countries:

    print("\n" + "  " * 70)
    print(country)
   

    print(
        working_data[country]["k30"]
        .value_counts(dropna=False)
        .sort_index()
    )

    values = set(
        working_data[country]["k30"]
        .dropna()
        .unique()
    )

    assert values.issubset({
        0, 1, 2, 3, 4
    }), (
        f"Unexpected k30 values in {country}: {values}"
    )

print(
    "\nk30 ordinal coding verification passed."
)


                                                                                                                                            
Kenya
k30
0    154
1    253
2    257
3    209
4    151
Name: count, dtype: int64

                                                                                                                                            
Tanzania
k30
0.0     94
1.0     96
2.0    182
3.0    109
4.0    118
NaN      1
Name: count, dtype: int64

                                                                                                                                            
Uganda
k30
0.0    168
1.0    168
2.0     88
3.0    118
4.0     60
NaN      3
Name: count, dtype: int64

k30 ordinal coding verification passed.


## 21. Clean Numeric Variables

The remaining explanatory variables are quantitative measures.

They are converted to numeric form while preserving valid zero values.

Non-numeric special responses are converted to missing values.



In [55]:

# CLEAN NUMERIC VARIABLES


for country in countries:

    df = working_data[country]

    for variable in numeric_variables:

        # Preserve original series
        s = df[variable].copy()

       

        s = s.astype("object")


        s = s.replace(
            SPECIAL_RESPONSES,
            np.nan
        )

       

        if variable == "b2b":

            s = s.replace({
                "Fully foreign owned (private)": 100
            })

      

        s = pd.to_numeric(
            s,
            errors="coerce"
        )

       
        # Store cleaned variable
        

        df[variable] = s

    working_data[country] = df

print(" Numeric variables cleaned successfully.")

 Numeric variables cleaned successfully.


## 22. Verify Numeric Variables

The numeric variables are inspected after conversion.

The checks focus on:

- numeric data type;
- missing observations;
- minimum and maximum values;
- unexpected negative values.

Valid zero values are retained.

In [56]:

# NUMERIC VARIABLE VERIFICATION


numeric_check_rows = []

for country in countries:

    df = working_data[country]

    for variable in numeric_variables:

        s = df[variable]

        numeric_check_rows.append({
            "Country": country,
            "Variable": variable,
            "dtype": str(s.dtype),
            "N": s.notna().sum(),
            "Missing": s.isna().sum(),
            "Min": s.min(),
            "Max": s.max()
        })

numeric_checks = pd.DataFrame(
    numeric_check_rows
)

display(
    numeric_checks
)

,Country,Variable,dtype,N,Missing,Min,Max
0,Kenya,l1,int64,1024,0,1.0,9.983000e+03
1,Kenya,d2,float64,1024,0,500000.0,2.200000e+11
2,Kenya,b7,float64,1021,3,2.0,6.900000e+01
3,Kenya,b2b,int64,1024,0,0.0,1.000000e+02
4,Tanzania,l1,float64,599,1,1.0,2.800000e+03
5,Tanzania,d2,float64,513,87,5000000.0,2.000000e+11
6,Tanzania,b7,float64,571,29,2.0,5.700000e+01
7,Tanzania,b2b,float64,597,3,0.0,1.000000e+02
8,Uganda,l1,int64,605,0,1.0,2.000000e+03
9,Uganda,d2,float64,559,46,2000000.0,6.000000e+12


## 23. Check for Impossible Negative Values

The variables used in the thesis represent quantities or percentages for which
negative values are not substantively meaningful.

Negative observations are therefore flagged for inspection rather than silently
deleted.

In [57]:

# NEGATIVE VALUE CHECK


negative_checks = []

for country in countries:

    df = working_data[country]

    for variable in numeric_variables:

        negative_count = int(
            (df[variable] < 0).sum()
        )

        negative_checks.append({
            "Country": country,
            "Variable": variable,
            "Negative_values": negative_count
        })

negative_checks = pd.DataFrame(
    negative_checks
)

display(
    negative_checks
)

,Country,Variable,Negative_values
0,Kenya,l1,0
1,Kenya,d2,0
2,Kenya,b7,0
3,Kenya,b2b,0
4,Tanzania,l1,0
5,Tanzania,d2,0
6,Tanzania,b7,0
7,Tanzania,b2b,0
8,Uganda,l1,0
9,Uganda,d2,0


## 24. Predictor Selection

After cleaning, the modelling predictors are explicitly selected.

Only the 12 thesis predictors and the constructed exporter target are retained
for the cleaned modelling dataset.

The original raw datasets remain untouched.

The resulting cleaned datasets will therefore contain:

- `exporter`;
- 12 predictors.

In [58]:

# SELECT CLEANED MODELLING VARIABLES


model_columns = [
    target
    if "target" in globals()
    else "exporter"
] + predictors

print("Final modelling columns:")
print(model_columns)

print("\nNumber of columns:")
print(len(model_columns))

Final modelling columns:
['exporter', 'l1', 'd2', 'b7', 'b2b', 'h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39', 'k30']

Number of columns:
13


## 25. Construct the Final Cleaned Datasets

The final cleaned datasets contain only the variables required for the
descriptive and modelling stages.

Rows are not removed solely because a predictor is missing.

Missing predictor values are retained for the next stage, where the modelling
preparation procedure will determine the appropriate treatment.

This separation prevents the descriptive-cleaning stage from introducing
model-specific preprocessing decisions.

In [59]:

# FINAL CLEANED DATASETS


clean_data = {}

for country in countries:

    df = working_data[country].copy()

    clean_df = df[
        model_columns
    ].copy()

    clean_data[country] = clean_df

    print("\n" + "  " * 70)
    print(country)
    

    print("Shape:", clean_df.shape)
    print("Columns:")
    print(clean_df.columns.tolist())


                                                                                                                                            
Kenya
Shape: (1024, 13)
Columns:
['exporter', 'l1', 'd2', 'b7', 'b2b', 'h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39', 'k30']

                                                                                                                                            
Tanzania
Shape: (600, 13)
Columns:
['exporter', 'l1', 'd2', 'b7', 'b2b', 'h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39', 'k30']

                                                                                                                                            
Uganda
Shape: (605, 13)
Columns:
['exporter', 'l1', 'd2', 'b7', 'b2b', 'h1', 'c22b', 'c36', 'e6', 'b3a', 'l10', 'c39', 'k30']


## 26. Final Missingness Assessment

Missing values are reported after the cleaning process.

This table provides the baseline missingness that will be passed to
`02_descriptive_analysis.ipynb`.

No imputation is performed here.

In [60]:

# FINAL MISSINGNESS


final_missing_rows = []

for country in countries:

    df = clean_data[country]

    for variable in model_columns:

        total = len(df)

        missing = int(
            df[variable].isna().sum()
        )

        final_missing_rows.append({
            "Country": country,
            "Variable": variable,
            "Total": total,
            "Missing": missing,
            "Missing_%": round(
                missing / total * 100,
                2
            )
        })

final_missingness = pd.DataFrame(
    final_missing_rows
)

display(
    final_missingness
)

,Country,Variable,Total,Missing,Missing_%
0,Kenya,exporter,1024,0,0.00
1,Kenya,l1,1024,0,0.00
2,Kenya,d2,1024,0,0.00
3,Kenya,b7,1024,3,0.29
4,Kenya,b2b,1024,0,0.00
5,Kenya,h1,1024,1,0.10
6,Kenya,c22b,1024,0,0.00
7,Kenya,c36,1024,0,0.00
8,Kenya,e6,1024,1,0.10
9,Kenya,b3a,1024,0,0.00


## 27. Check Duplicate Observations

Duplicate rows in the final modelling dataset are checked.

Duplicates are reported rather than automatically removed because duplicate
survey observations must only be removed when their origin is understood.

In [61]:

# DUPLICATE CHECK


for country in countries:

    df = clean_data[country]

    duplicate_count = int(
        df.duplicated().sum()
    )

    print(
        f"{country}: duplicate rows = {duplicate_count}"
    )

Kenya: duplicate rows = 1
Tanzania: duplicate rows = 1
Uganda: duplicate rows = 0


## 28. Final Structural Verification

The final cleaned datasets must contain:

- the `exporter` target;
- exactly 12 predictors;
- only valid binary values for binary variables;
- valid ordinal values for `k30`;
- numeric values for quantitative predictors.

Missing values are allowed and are reported separately.

In [63]:

# FINAL STRUCTURE VERIFICATION


for country in countries:

    df = clean_data[country]

    print("\n" + "  " * 70)
    print(country)
    

   
    # Target
   

    assert "exporter" in df.columns

    target_values = set(
        df["exporter"]
        .dropna()
        .unique()
    )

    assert target_values.issubset({
        0, 1
    }), (
        f"Invalid exporter values in {country}: "
        f"{target_values}"
    )

   
    # Predictors
    

    for variable in predictors:

        assert variable in df.columns, (
            f"{variable} missing in {country}"
        )

   
    # Binary variables
    

    for variable in binary_variables:

        values = set(
            df[variable]
            .dropna()
            .unique()
        )

        assert values.issubset({
            0, 1
        }), (
            f"Invalid binary values in "
            f"{country} - {variable}: {values}"
        )

  
    # Ordinal variable
    

    k30_values = set(
        df["k30"]
        .dropna()
        .unique()
    )

    assert k30_values.issubset({
        0, 1, 2, 3, 4
    })

    print(
        "Target:", "exporter"
    )

    print(
        "Predictors:",
        len(predictors)
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Missing values:",
        int(df.isna().sum().sum())
    )

    print(
        "Verification: PASSED"
    )


                                                                                                                                            
Kenya
Target: exporter
Predictors: 12
Rows: 1024
Missing values: 69
Verification: PASSED

                                                                                                                                            
Tanzania
Target: exporter
Predictors: 12
Rows: 600
Missing values: 438
Verification: PASSED

                                                                                                                                            
Uganda
Target: exporter
Predictors: 12
Rows: 605
Missing values: 266
Verification: PASSED


## 29. Save Cleaned Datasets

The cleaned country datasets are saved as Stata `.dta` files.

These files become the official input datasets for:

`02_descriptive_analysis.ipynb`

The raw WBES datasets are never overwritten.

In [64]:

# SAVE CLEANED DATASETS


for country in countries:

    output_path = (
        CLEAN_DIR /
        f"{country}_clean.dta"
    )

    clean_data[country].to_stata(
        output_path,
        write_index=False
    )

    print(
        f"{country} saved -> {output_path}"
    )

Kenya saved -> ../Data/clean/Kenya_clean.dta
Tanzania saved -> ../Data/clean/Tanzania_clean.dta
Uganda saved -> ../Data/clean/Uganda_clean.dta


## 30. Final Cleaning Summary

The final summary reports:

- original observations;
- cleaned observations;
- number of predictors;
- exporter observations;
- missing exporter observations;
- total missing predictor observations.

This provides a reproducible audit trail for the cleaning stage.

In [65]:

# FINAL CLEANING SUMMARY


cleaning_summary_rows = []

for country in countries:

    raw_df = raw_data[country]
    clean_df = clean_data[country]

    cleaning_summary_rows.append({
        "Country": country,
        "Raw_N": len(raw_df),
        "Clean_N": len(clean_df),
        "Predictors": len(predictors),
        "Exporters": int(
            (clean_df["exporter"] == 1).sum()
        ),
        "Non_exporters": int(
            (clean_df["exporter"] == 0).sum()
        ),
        "Missing_exporter": int(
            clean_df["exporter"].isna().sum()
        ),
        "Missing_predictor_values": int(
            clean_df[predictors]
            .isna()
            .sum()
            .sum()
        )
    })

cleaning_summary = pd.DataFrame(
    cleaning_summary_rows
)

display(
    cleaning_summary
)

,Country,Raw_N,Clean_N,Predictors,Exporters,Non_exporters,Missing_exporter,Missing_predictor_values
0,Kenya,1024,1024,12,179,845,0,69
1,Tanzania,600,600,12,86,504,10,428
2,Uganda,605,605,12,66,539,0,266
